# Postprocessing: Experiment Summary

Loads `summary.csv` from `results/<experiment_id>/` and writes LaTeX tables to `outputs/<experiment_id>/tables/`.

- Internal tables: CI length (vi), CI length (analytic), comparison (vi - analytic), status ok.
- Paper table: choose `paper_mode = 'vi'` or `'analytic'`.


In [1]:
import os
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)









In [2]:

import csv
import os
import numpy as np
from scipy.stats import binomtest

def load_summary(path):
    rows = []
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

def as_float(x):
    try:
        return float(x)
    except Exception:
        return float('nan')

def parse_n(row):
    try:
        return int(float(row.get('n')))
    except Exception:
        return None

def dgp_sort_key(s):
    try:
        parts = s.split('_')
        return int(parts[-1])
    except Exception:
        return s

def dgp_label(s):
    try:
        parts = s.split('_')
        idx = parts[-1]
        if len(parts) >= 2 and parts[1] == 'iv':
            return "\\mathcal{D}^{IV}_{%s}" % idx
        return "\\mathcal{D}_{%s}" % idx
    except Exception:
        return s


def orthogonal_cell(m):
    label = str(m).lower() if m is not None else ''
    if label.startswith('dr') or label in ('aipw', 'dr'):
        return r"\cmark"
    return r"\xmark"
def mean_std(values):
    arr = np.array([v for v in values if not np.isnan(v)], dtype=float)
    if arr.size == 0:
        return np.nan, np.nan
    mean = float(arr.mean())
    std = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
    return mean, std

def build_ci_stats(rows, methods, dgps, value_key):
    means = []
    stds = []
    for m in methods:
        row_means = []
        row_stds = []
        for d in dgps:
            vals = [as_float(r.get(value_key, '')) for r in rows if r.get('method') == m and r.get('dgp') == d]
            mean, std = mean_std(vals)
            row_means.append(mean)
            row_stds.append(std)
        means.append(row_means)
        stds.append(row_stds)
    return means, stds

def build_comparison_stats(rows, methods, dgps):
    means = []
    stds = []
    for m in methods:
        row_means = []
        row_stds = []
        for d in dgps:
            diffs = []
            for r in rows:
                if r.get('method') != m or r.get('dgp') != d:
                    continue
                vi = as_float(r.get('ci95_len_vi', ''))
                an = as_float(r.get('ci95_len_analytic', ''))
                if not (np.isnan(vi) or np.isnan(an)):
                    diffs.append(vi - an)
            mean, std = mean_std(diffs)
            row_means.append(mean)
            row_stds.append(std)
        means.append(row_means)
        stds.append(row_stds)
    return means, stds

def build_status_table(rows, methods, dgps):
    table = []
    for m in methods:
        row = []
        for d in dgps:
            subset = [r for r in rows if r.get('method') == m and r.get('dgp') == d]
            total = len(subset)
            ok = sum(1 for r in subset if r.get('status') == 'ok')
            cell = f"{ok}/{total}" if total > 0 else '0/0'
            row.append(cell)
        table.append(row)
    return table

def build_coverage_table(rows, methods, dgps, lo_key, hi_key, true_key):
    table = []
    for m in methods:
        row = []
        for d in dgps:
            subset = [r for r in rows if r.get('method') == m and r.get('dgp') == d]
            total = 0
            covered = 0
            for r in subset:
                lo = as_float(r.get(lo_key, ''))
                hi = as_float(r.get(hi_key, ''))
                true_val = as_float(r.get(true_key, ''))
                if np.isnan(lo) or np.isnan(hi) or np.isnan(true_val):
                    continue
                total += 1
                if lo <= true_val <= hi:
                    covered += 1
            frac = covered / total if total > 0 else float('nan')
            row.append(frac)
        table.append(row)
    return table

def build_coverage_stats(rows, methods, dgps, lo_key, hi_key, true_key):
    cov = []
    cov_lower = []
    cov_upper = []
    counts = []
    for m in methods:
        row_cov = []
        row_lower = []
        row_upper = []
        row_counts = []
        for d in dgps:
            subset = [r for r in rows if r.get('method') == m and r.get('dgp') == d]
            total = 0
            covered = 0
            for r in subset:
                lo = as_float(r.get(lo_key, ''))
                hi = as_float(r.get(hi_key, ''))
                true_val = as_float(r.get(true_key, ''))
                if np.isnan(lo) or np.isnan(hi) or np.isnan(true_val):
                    continue
                total += 1
                if lo <= true_val <= hi:
                    covered += 1
            if total > 0:
                p = covered / total
                ci = binomtest(covered, total).proportion_ci(confidence_level=0.95, method='exact')
                lower = float(ci.low)
                upper = float(ci.high)
            else:
                p = float('nan')
                lower = float('nan')
                upper = float('nan')
            row_cov.append(p)
            row_lower.append(lower)
            row_upper.append(upper)
            row_counts.append((covered, total))
        cov.append(row_cov)
        cov_lower.append(row_lower)
        cov_upper.append(row_upper)
        counts.append(row_counts)
    return cov, cov_lower, cov_upper, counts

def apply_mask_to_stats(means, stds, mask):
    masked_means = []
    masked_stds = []
    for i in range(len(means)):
        row_means = []
        row_stds = []
        for j in range(len(means[i])):
            row_means.append(means[i][j])
            row_stds.append(stds[i][j])
        masked_means.append(row_means)
        masked_stds.append(row_stds)
    return masked_means, masked_stds, mask

def format_coverage_cell(mean, lo, hi, is_best, fmt='{:.3f}', ci_fmt='{:.3f}'):
    if np.isnan(mean) or np.isnan(lo) or np.isnan(hi):
        return '$NA$'
    text = f"{fmt.format(mean)}"
    ci_text = f"\\scriptsize ({ci_fmt.format(lo)}-{ci_fmt.format(hi)})"
    if is_best:
        text = f"\\textbf{{{text}}}"
    return f"${text}$ {ci_text}"

def table_to_latex_coverage(methods, dgps, means, lowers, uppers, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}'):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\ '
    lines.append(header)
    lines.append(r'\hline')
    for i, m in enumerate(methods):
        row_cells = []
        for j, d in enumerate(dgps):
            col_vals = [means[k][j] for k in range(len(methods))]
            finite_vals = [v for v in col_vals if not np.isnan(v)]
            best = False
            if finite_vals:
                best_val = min(finite_vals, key=lambda v: abs(v - target))
                best = np.isclose(means[i][j], best_val, rtol=1e-9, atol=1e-12)
            row_cells.append(format_coverage_cell(means[i][j], lowers[i][j], uppers[i][j], best, fmt=fmt, ci_fmt=ci_fmt))
        lines.append(m.upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\ ')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def format_single(value, is_best, fmt='{:.3f}'):
    if np.isnan(value):
        return '$NA$'
    text = f"{fmt.format(value)}"
    if is_best:
        text = f"\\textbf{{{text}}}"
    return f"${text}$"

def table_to_latex_single(methods, dgps, values, best_mode='min', target=None, fmt='{:.3f}'):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\'
    lines.append(header)
    lines.append(r'\hline')
    for i, m in enumerate(methods):
        row_cells = []
        for j, d in enumerate(dgps):
            col_vals = [values[k][j] for k in range(len(methods))]
            finite_vals = [v for v in col_vals if not np.isnan(v)]
            best = False
            if finite_vals:
                if best_mode == 'closest' and target is not None:
                    best_val = min(finite_vals, key=lambda v: abs(v - target))
                    best = np.isclose(values[i][j], best_val, rtol=1e-9, atol=1e-12)
                else:
                    best_val = min(finite_vals)
                    best = np.isclose(values[i][j], best_val, rtol=1e-9, atol=1e-12)
            row_cells.append(format_single(values[i][j], best, fmt=fmt))
        lines.append(m.upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def format_cell(mean, std, is_best, fmt='{:.3f}', masked=False):
    if np.isnan(mean):
        return '$NA$'
    text = f"{fmt.format(mean)} ({fmt.format(std)})"
    if is_best:
        text = f"\\textbf{{{text}}}"
    if masked:
        text = f"\\textcolor{{gray}}{{\\cancel{{{text}}}}}"
    return f"${text}$"

def table_to_latex_ci(methods, dgps, means, stds, fmt='{:.3f}', mask=None):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\'
    lines.append(header)
    lines.append(r'\hline')
    for i, m in enumerate(methods):
        row_cells = []
        for j, d in enumerate(dgps):
            col_means = []
            for k in range(len(methods)):
                if mask is not None and not mask[k][j]:
                    continue
                col_means.append(means[k][j])
            finite_means = [v for v in col_means if not np.isnan(v)]
            best = False
            if finite_means:
                min_mean = min(finite_means)
                if mask is None or mask[i][j]:
                    best = np.isclose(means[i][j], min_mean, rtol=1e-9, atol=1e-12)
            masked = mask is not None and not mask[i][j]
            row_cells.append(format_cell(means[i][j], stds[i][j], best, fmt=fmt, masked=masked))
        lines.append(m.upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def table_to_latex_text(methods, dgps, table):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\'
    lines.append(header)
    lines.append(r'\hline')
    for m, row in zip(methods, table):
        row_cells = [f"${cell}$" for cell in row]
        lines.append(m.upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(text)
        f.write('\n')











In [3]:
experiment_id = 'exp1_backdoor_ate_attempt_2' # EXPERIMENT 1 for BACKDOOR (ATE)

# experiment_id = 'exp1_iv_ate_attempt_1' # EXPERIMENT 1 for IV
# experiment_id = 'exp1_iv_ate_attempt_1_TEST'

results_root = 'results'
outputs_root = 'outputs'
summary_path = os.path.join(results_root, experiment_id, 'summary.csv')
config_path = os.path.join(results_root, experiment_id, 'experiment_config.json')

if os.path.exists(config_path):
    import json
    with open(config_path, 'r', encoding='utf-8-sig') as f:
        config = json.load(f)
    print('Experiment config:')
    print(json.dumps(config, indent=2, sort_keys=True))
else:
    print('Experiment config not found:', config_path)

rows = load_summary(summary_path)
methods = sorted({r.get('method') for r in rows})
dgps = sorted({r.get('dgp') for r in rows}, key=dgp_sort_key)
dgps = [d for d in dgps if d != 'dgp_0']

n_list = sorted({parse_n(r) for r in rows if parse_n(r) is not None})

print('Methods:', methods)
print('DGPS:', dgps)
print('Sample sizes:', n_list)










Experiment config:
{
  "base_single_run": {
    "config_path": "configs/single_runs/single_run_config_example.json"
  },
  "experiment": {
    "diagnostics": true,
    "experiment_id": "exp1_backdoor_ate_attempt_2",
    "failure_mode": "fail_fast",
    "log": false,
    "overwrite": false,
    "plot": false,
    "progress": true,
    "save_artifacts": true
  },
  "sweep": {
    "dgp_list": [
      "dgp_1",
      "dgp_2",
      "dgp_3",
      "dgp_4",
      "dgp_5",
      "dgp_6",
      "dgp_7",
      "dgp_8",
      "dgp_9"
    ],
    "method_list": [
      "aipw",
      "ipw",
      "ra"
    ],
    "n_list": [
      100,
      1000
    ],
    "repetitions": 50,
    "run_seed0": 123
  }
}
Methods: ['aipw', 'ipw', 'ra']
DGPS: ['dgp_1', 'dgp_2', 'dgp_3', 'dgp_4', 'dgp_5', 'dgp_6', 'dgp_7', 'dgp_8', 'dgp_9']
Sample sizes: [100, 1000]


In [4]:
tables_dir_base = os.path.join(outputs_root, experiment_id, 'tables')

def tables_dir_for_n(n):
    if len(n_list) > 1:
        return os.path.join(tables_dir_base, f'n_{n}')
    return tables_dir_base

for n in n_list:
    rows_n = [r for r in rows if parse_n(r) == n]
    print(f'\n=== n={n} ({len(rows_n)} rows) ===')
    vi_means, vi_stds = build_ci_stats(rows_n, methods, dgps, 'ci95_len_vi')
    an_means, an_stds = build_ci_stats(rows_n, methods, dgps, 'ci95_len_analytic')
    cmp_means, cmp_stds = build_comparison_stats(rows_n, methods, dgps)
    status_ok = build_status_table(rows_n, methods, dgps)

    print('\nCI length (vi):', vi_means)
    print('\nCI length (analytic):', an_means)
    print('\nCI length diff (vi - analytic):', cmp_means)
    print('\nStatus ok:')
    print(status_ok)

    tables_dir = tables_dir_for_n(n)
    cov_vi, cov_vi_lower, cov_vi_upper, cov_vi_counts = build_coverage_stats(rows_n, methods, dgps, 'ci95_lo_vi', 'ci95_hi_vi', 'ate_true')
    cov_an, cov_an_lower, cov_an_upper, cov_an_counts = build_coverage_stats(rows_n, methods, dgps, 'ci95_lo_analytic', 'ci95_hi_analytic', 'ate_true')

    print('\nCoverage (vi):', cov_vi)
    print('\nCoverage (analytic):', cov_an)

    latex_cov_vi = table_to_latex_coverage(methods, dgps, cov_vi, cov_vi_lower, cov_vi_upper, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}')
    latex_cov_an = table_to_latex_coverage(methods, dgps, cov_an, cov_an_lower, cov_an_upper, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}')
    save_text(os.path.join(tables_dir, 'ci_coverage_vi.txt'), latex_cov_vi)
    save_text(os.path.join(tables_dir, 'ci_coverage_analytic.txt'), latex_cov_an)
    mask_vi = [[(cov_vi_upper[i][j] >= 0.95) if not np.isnan(cov_vi_upper[i][j]) else False for j in range(len(dgps))] for i in range(len(methods))]
    mask_an = [[(cov_an_upper[i][j] >= 0.95) if not np.isnan(cov_an_upper[i][j]) else False for j in range(len(dgps))] for i in range(len(methods))]

    vi_masked_means, vi_masked_stds, vi_mask = apply_mask_to_stats(vi_means, vi_stds, mask_vi)
    an_masked_means, an_masked_stds, an_mask = apply_mask_to_stats(an_means, an_stds, mask_an)

    latex_ci_length_masked_vi = table_to_latex_ci(methods, dgps, vi_masked_means, vi_masked_stds, mask=vi_mask)
    latex_ci_length_masked_an = table_to_latex_ci(methods, dgps, an_masked_means, an_masked_stds, mask=an_mask)
    save_text(os.path.join(tables_dir, 'ci_length_masked_vi.txt'), latex_ci_length_masked_vi)
    save_text(os.path.join(tables_dir, 'ci_length_masked_analytic.txt'), latex_ci_length_masked_an)
    latex_ci_vi = table_to_latex_ci(methods, dgps, vi_means, vi_stds)
    latex_ci_analytic = table_to_latex_ci(methods, dgps, an_means, an_stds)
    latex_ci_compare = table_to_latex_ci(methods, dgps, cmp_means, cmp_stds)
    latex_status_ok = table_to_latex_text(methods, dgps, status_ok)

    save_text(os.path.join(tables_dir, 'ci_length_vi.txt'), latex_ci_vi)
    save_text(os.path.join(tables_dir, 'ci_length_analytic.txt'), latex_ci_analytic)
    save_text(os.path.join(tables_dir, 'ci_length_compare.txt'), latex_ci_compare)
    save_text(os.path.join(tables_dir, 'status_ok.txt'), latex_status_ok)

    paper_mode = 'vi'  # set to 'vi' or 'analytic'
    if paper_mode == 'vi':
        paper_means, paper_stds = vi_means, vi_stds
    else:
        paper_means, paper_stds = an_means, an_stds
    latex_paper = table_to_latex_ci(methods, dgps, paper_means, paper_stds)
    save_text(os.path.join(tables_dir, 'ci_length_paper.txt'), latex_paper)

    print('Saved tables to:', tables_dir)












=== n=100 (1350 rows) ===

CI length (vi): [[0.7597559614658356, 0.8047015684366227, 0.8160617571830749, 0.8573495351076127, 0.7841205074310301, 0.7184234529614448, 1.4100769859790803, 0.8199681304693222, 1.6888782829761504], [1.3961491858959199, 1.4065017172336578, 1.5534333324909213, 1.7819254873752592, 1.2520990419626237, 0.9691961861371995, 1.8271943505764008, 1.0952225413799284, 3.7490606717109682], [0.22687232078909872, 0.30274343171715734, 0.316156851452589, 0.45005785334110265, 0.2200754099667072, 0.4561720300793648, 0.46998315196633345, 0.7783134761571883, 0.5103352627396583]]

CI length (analytic): [[0.7593069219001356, 0.8052515986092397, 0.815561748047831, 0.85582117174343, 0.7830996344545953, 0.719666405586496, 1.4103984766078366, 0.8198279989982648, 1.6936883045767275], [1.393556387896513, 1.4074539102960975, 1.5571428870308384, 1.7831404146467513, 1.2523921566936138, 0.9692559300414699, 1.8207924170754932, 1.09687540497063, 3.7518167701553553], [0.223605454682595, 0.302